# Exercise: Time-Series Modeling with statsmodels

A regional electric utility needs to forecast monthly electricity demand 12 months ahead. You have 8 years of monthly data with demand and temperature. Fit ARIMA and SARIMAX models and generate forecasts with prediction intervals.

- **Over-forecast** wastes fuel -- generators are spun up that aren't needed.
- **Under-forecast** risks brownouts -- insufficient capacity to meet peak demand.

Implement the three functions marked with `# YOUR CODE HERE`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
HORIZON = 12


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")
train_end = "2023-12-01"
train_df = df.loc[:train_end]
test_df = df.loc[train_end:].iloc[1:]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]
seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)
print(f"Data: {len(df)} months, {df.index[0].date()} to {df.index[-1].date()}")
print(f"Train: {len(train_df)} months, Test: {len(test_df)} months")


def fit_arima(train, order=(2, 1, 1)):
    """
    Fit a SARIMAX model (no seasonal or exogenous components).
    
    Parameters
    ----------
    train : pd.Series
        Training data with DatetimeIndex.
    order : tuple
        (p, d, q) order for the model.
    
    Returns
    -------
    statsmodels SARIMAXResults
        Fitted model results.
    """
    model = SARIMAX(train, order=order, trend=...)
    return model.fit(disp=False)

arima_fit = fit_arima(train_demand)
print(f"ARIMA AIC: {arima_fit.aic:.1f}")

def fit_sarimax(train, exog, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)):
    """
    Fit a SARIMAX model with seasonal components and exogenous variables.
    
    Parameters
    ----------
    train : pd.Series
    exog : pd.Series
        Exogenous variable (e.g., temperature).
    order : tuple
        (p, d, q) non-seasonal order.
    seasonal_order : tuple
        (P, D, Q, s) seasonal order.
    """
    model = SARIMAX(train, exog=exog, order=order, seasonal_order=...)
    return model.fit(disp=False)

sarimax_fit = fit_sarimax(train_demand, train_temp)
print(f"SARIMAX AIC: {sarimax_fit.aic:.1f}")

def forecast_with_intervals(fitted, steps, exog_future=None, alpha=0.05):
    """
    Generate forecast with prediction intervals.
    
    Parameters
    ----------
    fitted : statsmodels results
    steps : int
        Number of steps to forecast.
    exog_future : array-like, optional
        Future values of exogenous variable.
    alpha : float
        Significance level for intervals (default 0.05 = 95%).
    
    Returns
    -------
    tuple[pd.Series, pd.DataFrame]
        (predicted_mean, conf_int)
    """
    fc = fitted.get_forecast(steps=..., exog=...)
    return fc.predicted_mean, fc.conf_int(alpha=...)

arima_fc, arima_ci = forecast_with_intervals(arima_fit, HORIZON)
sarimax_fc, sarimax_ci = forecast_with_intervals(sarimax_fit, HORIZON, exog_future=future_temp.values)

In [ ]:
def forecast_with_intervals(fitted, steps, exog_future=None, alpha=0.05):
    """
    Generate forecast with prediction intervals.
    
    Parameters
    ----------
    fitted : statsmodels results
    steps : int
        Number of steps to forecast.
    exog_future : array-like, optional
        Future values of exogenous variable.
    alpha : float
        Significance level for intervals (default 0.05 = 95%).
    
    Returns
    -------
    tuple[pd.Series, pd.DataFrame]
        (predicted_mean, conf_int)
    """
    fc = fitted.get_forecast(steps=..., exog=...)
    return fc.predicted_mean, fc.conf_int(alpha=...)

arima_fc, arima_ci = forecast_with_intervals(arima_fit, HORIZON)
sarimax_fc, sarimax_ci = forecast_with_intervals(sarimax_fit, HORIZON, exog_future=future_temp.values)

In [ ]:
sarimax_fc, sarimax_ci = forecast_with_intervals(
    sarimax_fit, HORIZON, exog_future=future_temp.values
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for ax, name, fc, ci, color in [
    (axes[0], "ARIMA(2,1,1)", arima_fc, arima_ci, "tab:blue"),
    (axes[1], "SARIMAX + temperature", sarimax_fc, sarimax_ci, "tab:orange"),
]:
    ax.plot(train_demand.index, train_demand.values, color=UN["Black"], label="Historical")
    ax.plot(fc.index, fc.values, color=color, linestyle="--", label="Forecast")
    ax.fill_between(fc.index, ci.iloc[:, 0], ci.iloc[:, 1], color=color, alpha=0.15, label="95% interval")
    ax.set_ylabel("Demand (MWh)")
    ax.set_title(name)
    ax.legend(loc="upper left")
plt.tight_layout()
axes[0].set_ylabel("Value", fontsize=16)
for ax in axes:
    ax.tick_params(axis="both", labelsize=14)
plt.show()
